# Lab 3: Choosing Your Evaluator — SOLUTIONS
## CCI Session 7 — Lesson 3

**Duration:** 25 minutes

### Clinical Scenario
> The Oncology Summary Assistant takes a free-text pediatric oncology case and produces a structured tumor-board-ready summary (diagnosis, stage, histology, recommendation, monitoring). Yesterday in Lab 2 you built a 30-case dataset with `input`, `output`, and `human_label`. Today the AI Office needs you to decide *how* to evaluate model outputs at scale before deployment.

### Objective
Build and compare the **three evaluator families**:
1. **Functional evaluators** (deterministic, cheap, fast) — JSON schema + drug allowlist
2. **LLM-as-judge** (flexible, scalable, biased) — clinical relevance 1-5 with a careful rubric
3. **Human evaluation** (gold standard) — encoded in `human_label` from Lab 2

Then measure **cost**, **time**, and **disagreement**, and finally restructure the LLM judge as a **pairwise comparison**.

### Rule of thumb (Lesson 3)
> **Functional first. Judge second. Human as ground truth and final check.**

## Setup

In [ ]:
!pip install -q openai deepeval jsonschema pandas

In [ ]:
import os, re, json, time, random
from google.colab import userdata
from openai import OpenAI
import pandas as pd
from jsonschema import validate, ValidationError

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
client = OpenAI()
random.seed(42)

## Cell 1 — Load the Lab 2 dataset (30 Wilms tumor cases)

Each case has `case_id`, `input`, `output`, `human_label`.

In [ ]:
DATASET = [
    {'case_id': 1, 'input': '4-year-old female, palpable abdominal mass, no metastases on CT, unilateral right kidney lesion 8 cm, favorable histology on biopsy.', 'output': '{"diagnosis": "Wilms tumor", "stage": "I", "histology": "Favorable", "recommendation": "Nephrectomy followed by vincristine + actinomycin-D (EE-4A regimen)", "monitoring": "Chest X-ray and abdominal US every 3 months for 2 years"}', 'human_label': 'pass'},
    {'case_id': 2, 'input': '3-year-old male, right flank mass, lung nodules on CT, unilateral kidney tumor 11 cm, favorable histology.', 'output': '{"diagnosis": "Wilms tumor", "stage": "IV", "histology": "Favorable", "recommendation": "Nephrectomy + vincristine + actinomycin-D + doxorubicin (DD-4A) with whole-lung radiation", "monitoring": "CT chest every 3 months, abdominal US every 3 months for 2 years"}', 'human_label': 'pass'},
    {'case_id': 3, 'input': '5-year-old female, bilateral kidney masses on imaging, no metastases, biopsy pending.', 'output': '{"diagnosis": "Bilateral Wilms tumor", "stage": "V", "histology": "Pending", "recommendation": "Pre-operative chemotherapy with vincristine + actinomycin-D + doxorubicin, then nephron-sparing surgery", "monitoring": "Renal function (creatinine, GFR) and abdominal MRI every 3 months"}', 'human_label': 'pass'},
    {'case_id': 4, 'input': '2-year-old male, abdominal mass, biopsy shows diffuse anaplasia, unilateral 9 cm tumor, no metastases.', 'output': '{"diagnosis": "Wilms tumor", "stage": "II", "histology": "Diffuse anaplasia", "recommendation": "Nephrectomy + vincristine + doxorubicin + cyclophosphamide + carboplatin + etoposide (UH-2) with flank radiation", "monitoring": "Echocardiogram pre- and post-doxorubicin, CT chest/abdomen every 3 months"}', 'human_label': 'pass'},
    {'case_id': 5, 'input': '6-year-old female, unilateral 7 cm renal mass, favorable histology, lymph node positive at surgery, no distant mets.', 'output': '{"diagnosis": "Wilms tumor", "stage": "III", "histology": "Favorable", "recommendation": "Vincristine + actinomycin-D + doxorubicin (DD-4A) with flank radiation", "monitoring": "Abdominal US and CXR every 3 months for 2 years"}', 'human_label': 'pass'},
    {'case_id': 6, 'input': '18-month-old male, incidental abdominal mass, 4 cm unilateral kidney lesion, favorable histology, completely resected.', 'output': '{"diagnosis": "Wilms tumor", "stage": "I", "histology": "Favorable", "recommendation": "Nephrectomy alone (observation, EE-4A optional given young age and small tumor per AREN0532)", "monitoring": "Abdominal US every 3 months for 2 years"}', 'human_label': 'pass'},
    {'case_id': 7, 'input': '4-year-old female, large unilateral kidney mass with tumor thrombus extending into IVC, favorable histology, no distant mets.', 'output': '{"diagnosis": "Wilms tumor with IVC thrombus", "stage": "III", "histology": "Favorable", "recommendation": "Pre-operative chemotherapy (DD-4A) to shrink thrombus, then nephrectomy with thrombectomy and flank radiation", "monitoring": "Echocardiogram, CT chest/abdomen every 3 months"}', 'human_label': 'pass'},
    {'case_id': 8, 'input': '7-year-old male, recurrent Wilms tumor in lung 14 months post-treatment, originally Stage II favorable histology.', 'output': '{"diagnosis": "Recurrent Wilms tumor", "stage": "IV (relapse)", "histology": "Favorable", "recommendation": "Salvage with ifosfamide + carboplatin + etoposide (ICE) followed by high-dose chemo and autologous stem-cell rescue", "monitoring": "CT chest every 6 weeks during salvage, then every 3 months"}', 'human_label': 'pass'},
    {'case_id': 9, 'input': '3-year-old female, abdominal mass, biopsy shows clear cell sarcoma of the kidney, unilateral 8 cm tumor.', 'output': '{"diagnosis": "Clear cell sarcoma of the kidney", "stage": "II", "histology": "Clear cell sarcoma", "recommendation": "Nephrectomy + vincristine + doxorubicin + cyclophosphamide + etoposide (regimen I) with flank radiation", "monitoring": "MRI brain and bone scan at baseline, then CT chest/abdomen every 3 months"}', 'human_label': 'pass'},
    {'case_id': 10, 'input': '5-year-old male, unilateral 10 cm renal mass, favorable histology, peritoneal spillage during surgery, no distant mets.', 'output': '{"diagnosis": "Wilms tumor", "stage": "III", "histology": "Favorable", "recommendation": "Vincristine + actinomycin-D + doxorubicin (DD-4A) plus flank radiation", "monitoring": "Abdominal US and CXR every 3 months for 2 years"}', 'human_label': 'pass'},
    {'case_id': 11, 'input': '2-year-old male, palpable abdominal mass, 6 cm unilateral kidney lesion, favorable histology, no metastases.', 'output': '{"diagnosis": "Wilms tumor", "stage": "I", "histology": "Favorable", "recommendation": "Nephrectomy + vincristine + actinomycin-D (EE-4A)", "monitoring": "Abdominal US every 3 months for 2 years"}', 'human_label': 'pass'},
    {'case_id': 12, 'input': '4-year-old female, abdominal mass, focal anaplasia on biopsy, unilateral 12 cm tumor, completely resected.', 'output': '{"diagnosis": "Wilms tumor", "stage": "II", "histology": "Focal anaplasia", "recommendation": "Vincristine + actinomycin-D + doxorubicin (DD-4A)", "monitoring": "Echo pre- and post-doxorubicin, CT chest/abdomen every 3 months"}', 'human_label': 'pass'},
    {'case_id': 13, 'input': '6-year-old male, abdominal mass, biopsy shows malignant rhabdoid tumor of the kidney, no distant metastases.', 'output': '{"diagnosis": "Wilms tumor", "stage": "III", "histology": "Favorable", "recommendation": "Vincristine + actinomycin-D", "monitoring": "Abdominal US every 6 months"}', 'human_label': 'fail'},
    {'case_id': 14, 'input': '3-year-old female, unilateral 7 cm renal mass, favorable histology, no metastases, lymph nodes negative.', 'output': '{"diagnosis": "Wilms tumor", "stage": "II", "histology": "Favorable", "recommendation": "Nephrectomy + vincristine + actinomycin-D (EE-4A)", "monitoring": "Abdominal US and CXR every 3 months for 2 years"}', 'human_label': 'pass'},
    {'case_id': 15, 'input': '8-year-old male, abdominal mass, biopsy shows favorable histology Wilms, lung nodules on CT, unilateral 13 cm tumor.', 'output': '{"diagnosis": "Wilms tumor", "stage": "IV", "histology": "Favorable", "recommendation": "Vincristine + actinomycin-D + doxorubicin (DD-4A) with whole-lung radiation, escalation to regimen M (cyclophosphamide/etoposide) if incomplete lung response at week 6", "monitoring": "CT chest every 6 weeks during induction, then every 3 months"}', 'human_label': 'pass'},
    {'case_id': 16, 'input': '2-year-old female, incidental small 3 cm renal lesion, favorable histology, completely resected, age <24 months tumor weight <550g.', 'output': '{"diagnosis": "Wilms tumor", "stage": "I", "histology": "Favorable", "recommendation": "Nephrectomy alone with observation (very low risk per AREN0532)", "monitoring": "Abdominal US every 3 months for 2 years"}', 'human_label': 'pass'},
    {'case_id': 17, 'input': '5-year-old male, unilateral 9 cm renal mass, favorable histology, lymph nodes positive, no distant mets.', 'output': '{"diagnosis": "Wilms tumor", "stage": "III", "histology": "Favorable", "recommendation": "Recommend ibuprofen and follow up in 6 months", "monitoring": "None"}', 'human_label': 'fail'},
    {'case_id': 18, 'input': '4-year-old female, abdominal mass, diffuse anaplasia on biopsy, lung metastases on CT, unilateral 10 cm tumor.', 'output': '{"diagnosis": "Wilms tumor", "stage": "IV", "histology": "Diffuse anaplasia", "recommendation": "Vincristine + doxorubicin + cyclophosphamide + carboplatin + etoposide (UH-2) with flank and whole-lung radiation", "monitoring": "Echo pre- and post-doxorubicin, CT chest/abdomen every 6 weeks during induction"}', 'human_label': 'pass'},
    {'case_id': 19, 'input': '3-year-old male, bilateral renal masses on imaging, favorable histology on biopsy, no distant mets.', 'output': '{"diagnosis": "Bilateral Wilms tumor", "stage": "V", "histology": "Favorable", "recommendation": "Pre-operative vincristine + actinomycin-D + doxorubicin to enable nephron-sparing bilateral surgery", "monitoring": "Renal function and abdominal MRI every 3 months"}', 'human_label': 'pass'},
    {'case_id': 20, 'input': '6-year-old female, unilateral 8 cm renal mass, favorable histology, no metastases, completely resected stage II.', 'output': '{"diagnosis": "Wilms tumor", "stage": "II", "histology": "Favorable", "recommendation": "Vincristine + actinomycin-D (EE-4A)", "monitoring": "Abdominal US and CXR every 3 months for 2 years"}', 'human_label': 'pass'},
    {'case_id': 21, 'input': '4-year-old male, abdominal mass, favorable histology Wilms, peritoneal implants noted intraoperatively, no distant mets.', 'output': '{"diagnosis": "Wilms tumor", "stage": "III", "histology": "Favorable", "recommendation": "Vincristine + actinomycin-D + doxorubicin (DD-4A) with whole-abdomen radiation", "monitoring": "Abdominal US and CXR every 3 months for 2 years"}', 'human_label': 'pass'},
    {'case_id': 22, 'input': '5-year-old female, unilateral 11 cm renal mass, favorable histology, brain metastases on MRI.', 'output': '{"diagnosis": "Wilms tumor", "stage": "IV (CNS)", "histology": "Favorable", "recommendation": "DD-4A with whole-brain radiation; consider regimen M intensification", "monitoring": "MRI brain every 3 months, CT chest/abdomen every 3 months"}', 'human_label': 'pass'},
    {'case_id': 23, 'input': '2-year-old male, palpable mass, favorable histology, unilateral 5 cm tumor, completely resected, no metastases.', 'output': '{"diagnosis": "Wilms tumor", "stage": "I", "histology": "Favorable", "recommendation": "Nephrectomy + EE-4A (vincristine + actinomycin-D)", "monitoring": "Abdominal US every 3 months for 2 years"}', 'human_label': 'pass'},
    {'case_id': 24, 'input': '7-year-old male, unilateral 9 cm renal mass, focal anaplasia, no metastases, complete resection.', 'output': '{"diagnosis": "Wilms tumor", "stage": "II", "histology": "Focal anaplasia", "recommendation": "Vincristine + actinomycin-D + doxorubicin (DD-4A)", "monitoring": "Echo pre- and post-doxorubicin, CT chest/abdomen every 3 months"}', 'human_label': 'pass'},
    {'case_id': 25, 'input': '3-year-old female, abdominal mass, favorable histology, unilateral 8 cm tumor, lymph nodes positive, complete resection.', 'output': '{"diagnosis": "Wilms tumor", "stage": "III", "histology": "Favorable", "recommendation": "DD-4A with flank radiation", "monitoring": "Abdominal US and CXR every 3 months for 2 years"}', 'human_label': 'pass'},
    {'case_id': 26, 'input': '4-year-old male, unilateral 7 cm renal mass, favorable histology, lung nodules >1 cm on CT.', 'output': '{"diagnosis": "Wilms tumor", "stage": "IV", "histology": "Favorable", "recommendation": "DD-4A with whole-lung radiation", "monitoring": "CT chest every 6 weeks during induction, then every 3 months"}', 'human_label': 'pass'},
    {'case_id': 27, 'input': '5-year-old female, abdominal mass, favorable histology, unilateral 6 cm tumor, no metastases, lymph nodes negative.', 'output': '{"diagnosis": "Wilms tumor", "stage": "II", "histology": "Favorable", "recommendation": "EE-4A (vincristine + actinomycin-D)", "monitoring": "Abdominal US and CXR every 3 months for 2 years"}', 'human_label': 'pass'},
    {'case_id': 28, 'input': '6-year-old male, abdominal mass, biopsy diffuse anaplasia, unilateral 14 cm tumor, no metastases, complete resection stage III.', 'output': '{"diagnosis": "Wilms tumor", "stage": "III", "histology": "Diffuse anaplasia", "recommendation": "Vincristine + doxorubicin + cyclophosphamide + carboplatin + etoposide (UH-2) with flank radiation", "monitoring": "Echo pre- and post-doxorubicin, CT chest/abdomen every 3 months"}', 'human_label': 'pass'},
    {'case_id': 29, 'input': '3-year-old male, unilateral 10 cm renal mass, favorable histology, lung nodules on CT.', 'output': 'The patient has a kidney tumor. We recommend chemotherapy and surgery. Please follow up.', 'human_label': 'fail'},
    {'case_id': 30, 'input': '4-year-old female, unilateral 8 cm renal mass, favorable histology, no metastases, lymph nodes negative, complete resection.', 'output': '{"diagnosis": "Wilms tumor", "stage": "II", "histology": "Favorable", "recommendation": "EE-4A (vincristine + actinomycin-D)", "monitoring": "Abdominal US and CXR every 3 months for 2 years"}', 'human_label': 'pass'},
]

df = pd.DataFrame(DATASET)
print(f'Loaded {len(df)} cases. Pass={sum(df.human_label=="pass")}, Fail={sum(df.human_label=="fail")}')
df.head()

## Cell 2 — Functional evaluator #1: JSON schema validator

In [ ]:
SUMMARY_SCHEMA = {
    'type': 'object',
    'required': ['diagnosis', 'stage', 'histology', 'recommendation', 'monitoring'],
    'properties': {
        'diagnosis': {'type': 'string', 'minLength': 3},
        'stage': {'type': 'string', 'minLength': 1},
        'histology': {'type': 'string', 'minLength': 3},
        'recommendation': {'type': 'string', 'minLength': 10},
        'monitoring': {'type': 'string', 'minLength': 5},
    },
    'additionalProperties': True,
}

def eval_schema(output_text: str) -> bool:
    try:
        parsed = json.loads(output_text)
    except (json.JSONDecodeError, TypeError):
        return False
    try:
        validate(instance=parsed, schema=SUMMARY_SCHEMA)
    except ValidationError:
        return False
    return True

print('Case 1 valid? ', eval_schema(DATASET[0]['output']))   # True
print('Case 29 valid?', eval_schema(DATASET[28]['output']))  # False (free text)

## Cell 3 — Functional evaluator #2: drug allowlist

In [ ]:
ALLOWED_DRUGS = {
    'vincristine', 'actinomycin-d', 'actinomycin d', 'dactinomycin',
    'doxorubicin', 'etoposide', 'ifosfamide', 'carboplatin', 'cyclophosphamide',
}

KNOWN_DRUG_VOCAB = ALLOWED_DRUGS | {
    'cisplatin', 'methotrexate', 'irinotecan', 'topotecan', 'temozolomide',
    'rituximab', 'bevacizumab', 'paclitaxel', 'gemcitabine', '5-fluorouracil',
    'imatinib', 'sorafenib', 'pembrolizumab', 'ibuprofen', 'acetaminophen',
}

def eval_drugs(output_text: str) -> bool:
    text = output_text.lower()
    for drug in KNOWN_DRUG_VOCAB:
        pattern = rf'\b{re.escape(drug)}\b'
        if re.search(pattern, text):
            if drug not in ALLOWED_DRUGS:
                return False
    return True

print('Case 1 drugs ok? ', eval_drugs(DATASET[0]['output']))   # True
print('Case 17 drugs ok?', eval_drugs(DATASET[16]['output']))  # False (ibuprofen)

## Cell 4 — LLM-as-judge: clinical relevance (1-5)

In [ ]:
JUDGE_SYSTEM = """You are a pediatric oncology attending reviewing AI-generated tumor-board summaries.

Score the OUTPUT on CLINICAL RELEVANCE to the INPUT case using this rubric:
  5 = Fully appropriate. Diagnosis, stage, histology, treatment regimen, and monitoring all
      match standard COG/SIOP pediatric oncology practice for the case described.
  4 = Mostly appropriate. Minor issues (e.g., monitoring interval slightly off) but no clinical harm.
  3 = Partially appropriate. One major component is wrong or missing (e.g., wrong stage but right regimen).
  2 = Mostly inappropriate. Multiple components wrong, or treatment under-/over-intensified.
  1 = Unsafe or nonsensical. Wrong diagnosis, dangerous omission, or off-protocol agent.

STRICT INSTRUCTIONS to avoid bias:
- Judge ONLY clinical content, not writing style or length. A short correct answer scores higher than a long wrong one.
- Do NOT reward verbose explanations. Penalize padding.
- Do NOT consider whether the format is JSON; another evaluator handles that.
- If a drug is named that is not standard for Wilms tumor (e.g., ibuprofen for chemotherapy), score 1.
- Reason in 1-2 sentences, then output the integer score on the last line as: SCORE: <n>"""

FEW_SHOT = [
    {'role': 'user', 'content': (
        'INPUT: 4-year-old female, unilateral 6 cm renal mass, favorable histology, no metastases, lymph nodes negative.\n'
        'OUTPUT: {"diagnosis": "Wilms tumor", "stage": "I", "histology": "Favorable", '
        '"recommendation": "Nephrectomy + vincristine + actinomycin-D (EE-4A)", '
        '"monitoring": "Abdominal US every 3 months for 2 years"}'
    )},
    {'role': 'assistant', 'content': 'Stage I FH with EE-4A is standard COG. Monitoring interval correct.\nSCORE: 5'},
    {'role': 'user', 'content': (
        'INPUT: 5-year-old male, unilateral 9 cm renal mass, favorable histology, lymph nodes positive, no distant mets.\n'
        'OUTPUT: {"diagnosis": "Wilms tumor", "stage": "III", "histology": "Favorable", '
        '"recommendation": "Recommend ibuprofen and follow up in 6 months", "monitoring": "None"}'
    )},
    {'role': 'assistant', 'content': 'Stage III FH requires DD-4A plus flank radiation; ibuprofen is not chemotherapy and no monitoring is unsafe.\nSCORE: 1'},
]

def judge_clinical_relevance(input_text: str, output_text: str) -> int:
    messages = [{'role': 'system', 'content': JUDGE_SYSTEM}] + FEW_SHOT + [
        {'role': 'user', 'content': f'INPUT: {input_text}\nOUTPUT: {output_text}'}
    ]
    resp = client.chat.completions.create(
        model='gpt-4o',
        messages=messages,
        temperature=0,
        max_tokens=120,
    )
    text = resp.choices[0].message.content
    m = re.search(r'SCORE\s*:\s*([1-5])', text)
    return int(m.group(1)) if m else 3

print('Case 1 score:',  judge_clinical_relevance(DATASET[0]['input'],  DATASET[0]['output']))
print('Case 17 score:', judge_clinical_relevance(DATASET[16]['input'], DATASET[16]['output']))

## Cell 5 — Run all three evaluators on the 30 cases

In [ ]:
results = []
api_calls = 0

t0 = time.time()
for case in DATASET:
    schema_ok = eval_schema(case['output'])
    drugs_ok = eval_drugs(case['output'])
    judge_score = judge_clinical_relevance(case['input'], case['output'])
    api_calls += 1
    results.append({
        'case_id': case['case_id'],
        'human_label': case['human_label'],
        'schema_ok': schema_ok,
        'drugs_ok': drugs_ok,
        'judge_score': judge_score,
    })
elapsed = time.time() - t0

results_df = pd.DataFrame(results)
print(f'Total time: {elapsed:.1f}s   API calls: {api_calls}')
print(f'Avg judge time per case: {elapsed/len(DATASET):.2f}s')
results_df.head(10)

## Cell 6 — Cost, time, and disagreement analysis

In [ ]:
results_df['functional_pass'] = results_df['schema_ok'] & results_df['drugs_ok']
results_df['judge_pass'] = results_df['judge_score'] >= 4
results_df['human_pass'] = results_df['human_label'] == 'pass'

func_vs_human = (results_df['functional_pass'] == results_df['human_pass']).mean()
judge_vs_human = (results_df['judge_pass'] == results_df['human_pass']).mean()
func_vs_judge = (results_df['functional_pass'] == results_df['judge_pass']).mean()

print(f'Functional vs human agreement: {func_vs_human:.0%}')
print(f'Judge vs human agreement:      {judge_vs_human:.0%}')
print(f'Functional vs judge agreement: {func_vs_judge:.0%}')

# The dangerous quadrant: functional checks pass, but the judge sees a clinical problem.
silent_failures = results_df[results_df['functional_pass'] & (results_df['judge_score'] <= 3)]
print(f'\nSilent failures (functional PASS but judge <=3): {len(silent_failures)}')
print('These are exactly the cases that motivate using a judge in addition to functional checks.')
silent_failures

## Cell 7 — Pairwise judge: A vs B (with explicit position randomization)

In [ ]:
BASELINE_OUTPUTS = {
    1:  '{"diagnosis": "Kidney tumor", "stage": "early", "histology": "benign-looking", "recommendation": "Surgery", "monitoring": "Follow up"}',
    2:  '{"diagnosis": "Wilms tumor with lung mets", "stage": "advanced", "histology": "FH", "recommendation": "Chemo and surgery", "monitoring": "Imaging"}',
    3:  '{"diagnosis": "Bilateral kidney tumors", "stage": "V", "histology": "unknown", "recommendation": "Surgery first", "monitoring": "Labs"}',
    4:  '{"diagnosis": "Wilms tumor", "stage": "II", "histology": "anaplasia", "recommendation": "DD-4A", "monitoring": "CT scans"}',
    5:  '{"diagnosis": "Wilms tumor", "stage": "III", "histology": "FH", "recommendation": "Vincristine", "monitoring": "Imaging"}',
    6:  '{"diagnosis": "Wilms tumor", "stage": "I", "histology": "FH", "recommendation": "Surgery + chemo", "monitoring": "US"}',
    7:  '{"diagnosis": "Wilms tumor", "stage": "III", "histology": "FH", "recommendation": "Surgery then chemo", "monitoring": "Imaging"}',
    8:  '{"diagnosis": "Recurrent kidney tumor", "stage": "relapse", "histology": "FH", "recommendation": "Salvage chemo", "monitoring": "CT"}',
    9:  '{"diagnosis": "Kidney sarcoma", "stage": "II", "histology": "clear cell", "recommendation": "Surgery + chemo", "monitoring": "Imaging"}',
    10: '{"diagnosis": "Wilms tumor", "stage": "III", "histology": "FH", "recommendation": "DD-4A", "monitoring": "Imaging"}',
}

PAIRWISE_SYSTEM = """You are a pediatric oncology attending comparing two AI-generated tumor-board summaries
for the same patient case. Decide which summary is clinically better.

BIAS CONTROLS — read carefully:
- The labels A and B are randomly assigned. Do NOT prefer A or B by position.
- Length is irrelevant. Do not prefer the longer one. Penalize verbosity that adds no clinical value.
- Judge ONLY clinical correctness (diagnosis, stage, histology, regimen, monitoring) against COG/SIOP standards.
- If both are equally appropriate, output 'tie'. If one has a safety problem (e.g., wrong drug, wrong stage with treatment implications), the other wins regardless of polish.

Output exactly one line of the form: VERDICT: A   or   VERDICT: B   or   VERDICT: tie"""

def judge_pairwise(input_text: str, output_a: str, output_b: str) -> str:
    # Position randomization: with 50% probability, swap A and B before sending.
    swap = random.random() < 0.5
    sent_a, sent_b = (output_b, output_a) if swap else (output_a, output_b)
    user_msg = (
        f'INPUT: {input_text}\n\n'
        f'SUMMARY A:\n{sent_a}\n\n'
        f'SUMMARY B:\n{sent_b}'
    )
    resp = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {'role': 'system', 'content': PAIRWISE_SYSTEM},
            {'role': 'user', 'content': user_msg},
        ],
        temperature=0,
        max_tokens=60,
    )
    text = resp.choices[0].message.content
    m = re.search(r'VERDICT\s*:\s*(A|B|tie)', text, re.IGNORECASE)
    raw = m.group(1).lower() if m else 'tie'
    if raw == 'tie':
        return 'tie'
    # Map back to caller's original ordering
    if swap:
        return 'B' if raw == 'a' else 'A'
    return raw.upper()

print(judge_pairwise(DATASET[0]['input'], DATASET[0]['output'], BASELINE_OUTPUTS[1]))

## Cell 8 — Pairwise consistency check

In [ ]:
pairwise_rows = []
for cid in range(1, 11):
    case = DATASET[cid - 1]
    a, b = case['output'], BASELINE_OUTPUTS[cid]
    v1 = judge_pairwise(case['input'], a, b)
    v2 = judge_pairwise(case['input'], b, a)
    v2_flipped = {'A': 'B', 'B': 'A', 'tie': 'tie'}[v2]
    pairwise_rows.append({
        'case_id': cid,
        'verdict_1': v1,
        'verdict_2_flipped': v2_flipped,
        'consistent': v1 == v2_flipped,
    })

pw_df = pd.DataFrame(pairwise_rows)
print(f'Pairwise consistency across order-swapped runs: {pw_df["consistent"].mean():.0%}')
print('A consistent judge produces the same winner regardless of position.')
pw_df

## Cell 9 — Stretch: jury of 3 LLM judges

In [ ]:
JURY_PROMPTS = {
    'strict':   JUDGE_SYSTEM + '\n\nWhen in doubt between two scores, choose the LOWER one. Be skeptical.',
    'lenient':  JUDGE_SYSTEM + '\n\nGive benefit of the doubt for minor wording. Focus on whether the patient would receive correct care.',
    'safety':   JUDGE_SYSTEM + '\n\nFocus exclusively on patient safety. Anything that could lead to harm scores 1 or 2 regardless of other strengths.',
}

def judge_with_prompt(system_prompt: str, input_text: str, output_text: str) -> int:
    messages = [{'role': 'system', 'content': system_prompt}] + FEW_SHOT + [
        {'role': 'user', 'content': f'INPUT: {input_text}\nOUTPUT: {output_text}'}
    ]
    resp = client.chat.completions.create(model='gpt-4o', messages=messages, temperature=0, max_tokens=120)
    m = re.search(r'SCORE\s*:\s*([1-5])', resp.choices[0].message.content)
    return int(m.group(1)) if m else 3

def jury_vote(input_text: str, output_text: str) -> str:
    scores = [judge_with_prompt(p, input_text, output_text) for p in JURY_PROMPTS.values()]
    passes = sum(1 for s in scores if s >= 4)
    return 'pass' if passes >= 2 else 'fail'

# Run on a few cases (full 30 would triple cost)
sample = DATASET[:5] + [DATASET[16], DATASET[28]]  # include the two known-bad cases
jury_compare = []
for c in sample:
    jury_compare.append({
        'case_id': c['case_id'],
        'human': c['human_label'],
        'jury': jury_vote(c['input'], c['output']),
    })
pd.DataFrame(jury_compare)

## KHCC Connection

Every clinical AI tool deployed at KHCC needs an evaluation strategy before it sees a patient. The three-evaluator-family approach you practiced today is the **foundation** of that process:

- **Functional evaluators** are your CI/CD smoke tests — they catch broken JSON, missing fields, off-protocol drugs in milliseconds and run on every code change.
- **LLM-as-judge** is your scalable QA — it lets you measure clinical relevance across hundreds or thousands of cases without burning attending time. But it must be carefully prompted with bias controls, and treated with skepticism.
- **Human evaluation** remains the ground truth and the final sign-off. The other two exist to make the human's time go further, not to replace it.

When you build the next clinical assistant for the AI Office, start with a small human-labeled set, write functional checks for everything you can express as a rule, and only reach for an LLM judge when the criterion is genuinely subjective. That is how the AI Office signs off on clinical AI.